# Stellar occultation predictions with the Solar System Portal

- **Workshop:** For the **IDAC-BR V** hands-on workshop at the 3rd LSST@LATAM Meeting (ICTP-SAIFR).
- **Duration:** 15 min (run this notebook before `tutorial_tap.ipynb`).
- **Learning objective:** Retrieve predicted stellar occultations with `lineaSSP`, keep those whose shadow passes near São Paulo, and display one shadow-path map.
- **Tools:** [LIneA Occultation Prediction Database](https://solarsystem.linea.org.br) (public), [REST API](https://solarsystem.linea.org.br/docs/user-guide/api/) (public), [`lineaSSP`](https://github.com/linea-it/lineaSSP) (public).
- **Data products:** public occultation predictions (Gaia DR3 stars and updated orbits). This notebook does not use SQL, TAP, or LSST data rights.
- **Prerequisites:** A LIneA account ([register here](https://register.linea.org.br)) if using LIneA JupyterHub (the portal and API themselves require no login), plus basic knowledge of Python, Jupyter, and RA/Dec.
- **Credit:** [lineaSSP README](https://github.com/linea-it/lineaSSP), [LOPD API user guide](https://solarsystem.linea.org.br/docs/user-guide/api/), [LIneA occultation documentation](https://docs.linea.org.br/en/sci-platforms/linea-occulation-prediction-database.html), RTN-045, DOI [10.11578/rubin/dc.20250909.20](https://doi.org/10.11578/rubin/dc.20250909.20).
- **Get Support:** LIneA Science Platform forum (`help`) and [data.linea.org.br](https://data.linea.org.br).

**This notebook is HTTP only.** Catalogue queries (`mpc_sbn`, `stn='X05'`, DP1/DP2) are in `tutorial_tap.ipynb`.

### Agenda

| Section | Tool |
| --- | --- |
| 1. Getting ready | Portal / `lineaSSP` |
| 2. One body | `Prediction.by_name` |
| 3. Query parameters | REST / `get_data` |
| 4. Date window and site filter | `get_data` / `geofilter` |
| 5. Shadow map | `generate_map` |
| 6. Exercises, wrap-up, and Q&A | `lineaSSP` |

**Related tutorials:** [lineaSSP README](https://github.com/linea-it/lineaSSP), LIneA [Occultation Prediction Database](https://docs.linea.org.br/en/sci-platforms/linea-occulation-prediction-database.html), and `tutorial_tap.ipynb`.

The first code cell has `USE_FALLBACK`. Leave it `False` to call the portal. Set it `True`, or run the fallback cell under a failed live cell, to load `data/occultations_sample.csv` and the packaged map.

> **Figure 1:** Occultation Prediction Database public home page.
>
> ![Occultation Prediction Database home page, with the forecast totals and the date-range filter](figures/occultation-db-home.png)
>
> *Alt-text: Browser view of solarsystem.linea.org.br showing the LIneA header, the total-forecast counters, and the start of the date-range filter.*

## 1. Getting ready

1. Open the [Occultation Prediction Database](https://solarsystem.linea.org.br). Predictions are public; no login.
2. Run this notebook on Colab (upload this folder) or on the LIneA JupyterHub image **SolarSystem**.
3. The examples below follow the [`lineaSSP` README](https://github.com/linea-it/lineaSSP/blob/main/README.md): `Prediction()`, `by_name`, `get_data`, `geofilter`, `generate_map`.

> **Checkpoint:** If `Prediction()` fails or the portal returns 5xx, set `USE_FALLBACK = True` and **Run all**. Tables come from `data/occultations_sample.csv`; the map from `data/figures/occultation_path_map.png`.

In [ ]:
%matplotlib inline

from pathlib import Path

import pandas as pd
from IPython.display import Image, display

try:
    from lineaSSP import Prediction, geofilter, generate_map
except ImportError:
    Prediction = None
    geofilter = None
    generate_map = None

# True: skip live calls and use the fallback cell under each section.
USE_FALLBACK = False

# Key columns for table previews (full API/CSV tables have ~100 columns).
COLS = [
    "name",
    "date_time",
    "ra_star_candidate",
    "dec_star_candidate",
    "g_star",
    "magnitude_drop",
    "event_duration",
    "diameter",
]

api = None
pluto = None
near = []
near_df = None
map_shown = False
if USE_FALLBACK:
    print("Skipped: USE_FALLBACK is True.")
elif Prediction is None:
    print("lineaSSP is not installed. pip install lineassp, or set USE_FALLBACK = True.")
else:
    try:
        api = Prediction()
        print(api)
    except Exception as exc:
        print(f"Prediction() failed ({exc}). Run the fallback cell below.")


In [ ]:
# Fallback: later cells load data/occultations_sample.csv when api is None.
if api is None:
    print("No Prediction client. Fallback cells will read data/occultations_sample.csv.")
else:
    print(api)


## 2. One body with `by_name`

Same call as the README (`limit='all'` follows pagination; default cap is 1000 rows). Pluto is the worked example so the packaged CSV matches. For the 2024-09-01 05:41:16 UT event the star is at RA `20 10 18.2758`, Dec `-23 24 20.706`.

Confirm the table has at least one Pluto row and a `date_time` column.

> **Checkpoint:** If this cell fails, run the fallback. The 2026-10-20 snapshot has **17** Pluto rows.

In [ ]:
if USE_FALLBACK or api is None:
    print("Skipped. Run the fallback cell below.")
else:
    try:
        predictions = api.by_name("Pluto", limit="all")
        pluto = pd.DataFrame(predictions)
        display(pluto[[c for c in COLS if c in pluto.columns]].head())
        print("Pluto events:", len(pluto))
    except Exception as exc:
        pluto = None
        print(f"Live by_name failed ({exc}). Run the fallback cell below.")


In [ ]:
import utils

if pluto is None:
    pluto = utils.load_fallback_occultations()
    display(pluto[[c for c in COLS if c in pluto.columns]].head())
    print("Fallback | Pluto events:", len(pluto))
else:
    print(f"Live Pluto table kept: {len(pluto)} rows.")


## 3. Query parameters (`lineaSSP` and REST)

`Prediction.get_data(params=…)` sends the same query string as `GET https://solarsystem.linea.org.br/api/occultations/`. The names below are the LOPD API filters (`OccultationFilter`). Full lists: [API user guide](https://solarsystem.linea.org.br/docs/user-guide/api/), [Python module](https://solarsystem.linea.org.br/docs/user-guide/python_module/), [Swagger](https://solarsystem.linea.org.br/api/docs/), [ReDoc](https://solarsystem.linea.org.br/api/redoc/), [OpenAPI schema](https://solarsystem.linea.org.br/api/schema).

`by_name("Pluto")` is `params={"name": "Pluto"}`. Quota: 100 requests per hour, 1000 per day. The JSON page is `count`, `next`, `previous`, `results` (default 100 rows per page). `lineaSSP` walks pages for you (`limit='all'` or an integer).

| Parameter | What it selects |
| --- | --- |
| `name`, `number` | Body name or MPC number. Comma-separated lists are allowed (`name=Chiron,Eris`). |
| `date_time_after`, `date_time_before` | Event UTC window (ISO 8601, with or without time). |
| `magnitude_min`, `magnitude_max` | Gaia **G of the star** (`g_star`). Larger G is a fainter star. |
| `nightside` | Keep events whose path is on Earth's night side. |
| `local_solar_time_after`, `local_solar_time_before` | Local solar time at the event (`HH:MM`). A window that crosses midnight is valid (`21:00` → `03:00`). |
| `base_dynclass`, `dynclass` | Skybot class / subclass (exact, case-insensitive), e.g. `KBO`, `KBO>Resonant>5:2`. |
| `diameter_min`, `diameter_max` | Body diameter (km). |
| `event_duration_min`, `event_duration_max` | Maximum duration of the occultation (s). |
| `magnitude_drop_min`, `magnitude_drop_max` | Expected drop in stellar magnitude. |
| `closest_approach_uncertainty_km_min`, `closest_approach_uncertainty_km_max` | Geocentric closest-approach uncertainty (km). |
| `latitude`, `longitude`, `location_radius` | Server-side geo cut (km). Experimental on the API. `geofilter` does the same cut **after** download. |
| `hash_id`, `jobid` | One prediction, or one prediction job. |

`geofilter(data, latitude, longitude, radius)` is **not** a query parameter. It filters the list already in memory.

The next cell runs one page of REST with the same `params` dict that `get_data` would send. Confirm `count` is at least as large as `len(results)`.

In [ ]:
import requests

params = {
    "name": "Pluto",
    "date_time_after": "2026-10-01T00:00:00Z",
    "date_time_before": "2026-11-30T23:59:59Z",
    "magnitude_max": 15,
    "nightside": True,
}

# lineaSSP uses the same dict:
# events = api.get_data(params=params, limit="all")

url = "https://solarsystem.linea.org.br/api/occultations/"
if USE_FALLBACK:
    print("Skipped: USE_FALLBACK is True.")
else:
    try:
        response = requests.get(url, params=params, timeout=60)
        response.raise_for_status()
        page = response.json()
        print("REST", response.url)
        print("count =", page["count"], "| this page =", len(page["results"]), "| next =", page.get("next"))
    except Exception as exc:
        print(f"REST query failed ({exc}).")


## 4. Date window and `geofilter`

`get_data(params=…)` is the README date-range query. `geofilter` keeps events whose shadow passes near a site. Coordinates match the README: São Paulo, latitude `-23.55`, longitude `-46.63`.

The workshop uses a 2000 km radius so the packaged snapshot is not empty. Confirm the printed count is at least 1.

In [ ]:
if USE_FALLBACK or api is None or geofilter is None:
    print("Skipped. Run the fallback cell below.")
else:
    params = {
        "date_time_after": "2026-10-01T00:00:00Z",
        "date_time_before": "2026-11-30T23:59:59Z",
        "magnitude_max": 15,
        "nightside": True,
    }
    try:
        events = api.get_data(params=params, limit=50)
        near = geofilter(events, latitude=-23.55, longitude=-46.63, radius=2000)
        near_df = pd.DataFrame(near)
        display(near_df[[c for c in COLS if c in near_df.columns]].head())
        print("Events within 2000 km of São Paulo:", len(near))
    except Exception as exc:
        near = []
        near_df = None
        print(f"Live get_data / geofilter failed ({exc}). Run the fallback cell below.")


In [ ]:
import ast

import utils

if near_df is None:
    near_df = utils.load_fallback_occultations()
    records = near_df.to_dict(orient="records")
    for row in records:
        coeff = row.get("occ_path_coeff")
        if isinstance(coeff, str):
            try:
                row["occ_path_coeff"] = ast.literal_eval(coeff)
            except (SyntaxError, ValueError):
                pass
    if geofilter is not None:
        try:
            near = geofilter(records, latitude=-23.55, longitude=-46.63, radius=2000)
            near_df = pd.DataFrame(near)
            print("Fallback | after geofilter (2000 km of São Paulo):", len(near))
        except Exception as exc:
            near = records
            print(f"Fallback | packaged events (geofilter skipped: {exc}):", len(near_df))
    else:
        near = records
        print("Fallback | packaged events (geofilter unavailable):", len(near_df))
    display(near_df[[c for c in COLS if c in near_df.columns]].head())
else:
    print(f"Live São Paulo sample kept: {len(near_df)} rows.")


## 5. Shadow map with `generate_map`

`generate_map` wraps [SORA](https://sora.readthedocs.io). Cite SORA if you use the figure. Pass one (or a few) events; the function iterates the list.

Confirm a path is drawn for the named body and `date_time`.

> **Accessibility note:** The map encodes geography. Do not rely on color alone to read the path.

In [ ]:
figdir = Path("data/figures")
figdir.mkdir(parents=True, exist_ok=True)
map_path = figdir / "occultation_path_map.png"
map_shown = False

if USE_FALLBACK or api is None or generate_map is None or not near:
    print("Skipped. Run the fallback cell below.")
else:
    try:
        generate_map(data=near[:1], dpi=120, path=str(figdir) + "/", nameimg="occultation_path_map")
        if map_path.exists():
            display(Image(filename=str(map_path)))
            map_shown = True
        print("Confirm the shadow path is drawn for", near[0].get("name"), "at", near[0].get("date_time"))
    except Exception as exc:
        print(f"Live generate_map failed ({exc}). Run the fallback cell below.")


In [ ]:
# Fallback: packaged PNG. Skip if the live cell already showed a map.
map_path = Path("data/figures/occultation_path_map.png")
if not map_shown:
    if map_path.exists():
        display(Image(filename=str(map_path)))
        print("Confirm the shadow path is drawn for Pluto at 2024-09-01 05:41:16 UTC")
    else:
        print("Packaged map missing: data/figures/occultation_path_map.png")
else:
    print("Live map kept.")


> **Figure 2:** Shadow path of one occultation on Earth.
>
> *Alt-text: World map with the predicted shadow path and the event label for the selected body and UTC time.*

## Exercises

Finish after the guided cells, or later if the portal is slow.

**Exercise 1 (`by_name`).** Copy the README pattern for Chariklo: `api.by_name("Chariklo", limit="all")`, keep `g_star < 15`, write `chariklo_g15.csv`, then `geofilter` for São Paulo with `radius=1500`. Report how many rows remain.

A live count on 2026-10-20 returned **46** Chariklo events before the G cut. Zero rows after `geofilter` is a valid geographic result. The packaged snapshot has no Chariklo rows.

> *Hint:* Use the cell below. Confirm `chariklo_g15.csv` exists and print the `geofilter` count.

**Exercise 2 (`get_data` / REST).** Change `magnitude_max` to `12` in the section 3 `params` (or in the REST cell) and rerun. Confirm the new `count` is smaller than with `magnitude_max=15`.

**Exercise 3 (`generate_map`).** Call `generate_map` on the first event (`near[:1]` or `near_df.iloc[:1].to_dict(orient="records")`) with `dpi=120`. Confirm a PNG appears in `data/figures/`.

In [ ]:
# Exercise 1 — README pattern: by_name + DataFrame + geofilter
body = "Chariklo"

if USE_FALLBACK or api is None or geofilter is None:
    print("No Prediction client. Install lineaSSP / set USE_FALLBACK = False when the portal is up.")
else:
    try:
        df = pd.DataFrame(api.by_name(body, limit="all"))
        df["g_star"] = pd.to_numeric(df["g_star"], errors="coerce")
        chariklo_g15 = df.loc[df["g_star"] < 15].copy()
        chariklo_g15.to_csv("chariklo_g15.csv", index=False)
        print(body, "before G cut:", len(df), "| after:", len(chariklo_g15))
        near_chariklo = geofilter(
            chariklo_g15.to_dict(orient="records"),
            latitude=-23.55,
            longitude=-46.63,
            radius=1500,
        )
        print("After geofilter (1500 km of São Paulo):", len(near_chariklo))
    except Exception as exc:
        print(f"Exercise 1 query failed ({exc}).")


## Wrap-up and Q&A

You called `Prediction.by_name`, sent the same `params` on REST and with `get_data`, then `geofilter` and `generate_map`. TAP SQL starts in `tutorial_tap.ipynb`.

Review: a Pluto table, a São Paulo subset, a shadow-path PNG, and `chariklo_g15.csv` if Exercise 1 ran live.

Questions: Did the portal require a login? What did a failed `Prediction()` look like?

## References

- `lineaSSP` — [GitHub README](https://github.com/linea-it/lineaSSP) · [Python module](https://solarsystem.linea.org.br/docs/user-guide/python_module/) · [PyPI](https://pypi.org/project/lineaSSP/)
- LOPD REST API — [user guide](https://solarsystem.linea.org.br/docs/user-guide/api/) · [Swagger](https://solarsystem.linea.org.br/api/docs/) · [ReDoc](https://solarsystem.linea.org.br/api/redoc/) · [OpenAPI schema](https://solarsystem.linea.org.br/api/schema)
- Occultation Prediction Database — [solarsystem.linea.org.br](https://solarsystem.linea.org.br) · [LIneA documentation](https://docs.linea.org.br/en/sci-platforms/linea-occulation-prediction-database.html)
- SORA — [sora.readthedocs.io](https://sora.readthedocs.io)
- RTN-045 · DOI [10.11578/rubin/dc.20250909.20](https://doi.org/10.11578/rubin/dc.20250909.20)